# Jev as a routing-failure judge

Check whether the router failed to interpret an indirect or underspecified request.
Use the latest rules and data: 294 cases after excluding six prompt examples.
This notebook needs only a TypeSafe API key. For the independent LLM and escalation,
use [the comparison notebook](llm_as_judge.ipynb).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Colab needs a checkout; locally, open this notebook inside the repository.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    checkout = Path("/content/routing-evals-lab")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "codex/jev-routing-judge",
                        "https://github.com/The-Gen-Academy/Mastering-Agentic-AI-Week4-Session1.git",
                        str(checkout)], check=True)
    os.chdir(checkout)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "python-dotenv"], check=True)

folders = [Path.cwd(), Path("routing_agent_evals"), Path("AI-Evals-Principles/routing_agent_evals")]
LAB = next(folder.resolve() for folder in folders if (folder / "judge_lab.py").exists())
os.chdir(LAB)
sys.path.insert(0, str(LAB))

In [ ]:
import os
from getpass import getpass
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from judge_lab import JudgeClient, load_dataset, batch, metrics, always_false
from prompts.llm_as_judge_prompt import JEV_JUDGE_PROMPT

load_dotenv(LAB / ".env")
MODE = os.environ.get("JEV_MODE", "live")  # Use replay to read the saved cache without API calls.
assert MODE in {"live", "replay"}
RUN = Path(os.environ.get("JEV_RUN_DIR", LAB / ".runs/indirect-jev"))
key = os.environ.get("TYPESAFE_API_KEY")
if MODE == "live" and not key and IN_COLAB:
    from google.colab import userdata
    try:
        key = userdata.get("TYPESAFE_API_KEY")
    except Exception:
        pass
if MODE == "live" and not key:
    key = getpass("TypeSafe API key (hidden): ")
client = JudgeClient(RUN / "cache", live=(MODE == "live"), api_keys={"jev": key})
client.prompts["jev"] = JEV_JUDGE_PROMPT

## Load and judge

Only the request and predicted intent reach Jev. It chooses TRUE when the selected
failure is established, otherwise FALSE. Human labels are used only for comparison.

In [ ]:
rows = load_dataset(LAB / "data")
data = pd.DataFrame(rows)
print(f"{len(data)} cases: {data.human_label.eq('TRUE').sum()} selected failures, "
      f"{data.human_label.eq('FALSE').sum()} other cases.")
display(data.head(3))

In [ ]:
jev = batch(rows, lambda row: client.call(row, "jev"))
display(pd.DataFrame([jev[rows[0]["id"]]]).reindex(columns=["label", "confidence", "probabilities", "error"]))

## Compare with the references

Check TRUE recall and false alarms, not just agreement. Missing decisions remain
unresolved. Costs may be estimates; unknown costs are shown as unavailable.

In [ ]:
comparison = pd.DataFrame([
    metrics(rows, answers, name)
    for name, answers in {"Always FALSE": always_false(rows), "Jev": jev}.items()
])
comparison["cost_usd"] = comparison.cost_usd.where(comparison.unknown_cost_attempts.eq(0))
display(comparison[["method", "accuracy", "precision_TRUE", "recall_TRUE", "f1_TRUE", "FP", "unresolved", "cost_usd"]].round(4))
RUN.mkdir(parents=True, exist_ok=True)
comparison.to_csv(RUN / "comparison.csv", index=False)
pd.DataFrame(jev.values()).to_csv(RUN / "results.csv", index=False)